# Sección 8. Combinación de DataFrames

En esta sección exploraremos la posibilidad de combinar información de DataFrames separados en un único DataFrame. Para ello veremos distintas formas de combinar los DataFrames de manera que la información conserve su coherencia e integración dentro de ese proceso.

## 8.1. Concatenación de DataFrames en Pandas

__Def. Concatenar__ DataFrames significa unir estructuras de datos a lo largo de un eje. 

Los DataFrames, como ya hemos visto, son estructuras bidimensionales por lo que tendremos dos opciones para concatenar:

* Unir filas -> Concatenación vertical
* Unir columnas -> Concatenación horizontal

La concatenación resulta útil cuando tenemos datos iguales separados por años o meses, queremos combinar tablas relacionadas o necesitamos añadir nuevos datos a un dataset.

Imaginemos por ejemplo que tenemo un DataFrame con los datos de ventas de la región sur y otro con los datos de ventas de la región norte:

In [1]:
import pandas as pd

df_norte = pd.DataFrame({
    "ciudad": ["Oviedo", "Avilés", "A Coruña"],
    "ventas": [200, 150, 180]
})

df_sur = pd.DataFrame({
    "ciudad": ["Sevilla", "Cádiz", "Málaga", "Córdoba"],
    "ventas": [120, 140, 130, 90]
})

display(df_norte)
display(df_sur)

,ciudad,ventas
0,Oviedo,200
1,Avilés,150
2,A Coruña,180


,ciudad,ventas
0,Sevilla,120
1,Cádiz,140
2,Málaga,130
3,Córdoba,90


En este caso la concatenación que tiene sentido es la concatenación vertical, es decir, crear un único DataFrame con las columnas `ciudad` y `ventas` formadas por la combinación de dichas columnas en cada tabla:

In [2]:
df_ciudad_concat = pd.concat([df_norte, df_sur])
df_ciudad_concat

,ciudad,ventas
0,Oviedo,200
1,Avilés,150
2,A Coruña,180
0,Sevilla,120
1,Cádiz,140
2,Málaga,130
3,Córdoba,90


__Nota.__ Es importante observar que aunque la concatenación se ha realizado con éxito hemos conservado los índices lo que ha provocado que el índice ya no sea único. 

Si deseamos evitar esto podemos utilizar el argumento `ignore_index=True`, lo que generará un nuevo índice continuo:

In [3]:
df_ciudad_concat = pd.concat([df_norte, df_sur], ignore_index=True)
df_ciudad_concat

,ciudad,ventas
0,Oviedo,200
1,Avilés,150
2,A Coruña,180
3,Sevilla,120
4,Cádiz,140
5,Málaga,130
6,Córdoba,90


Si los DataFrames no comparten las mismas columnas, la concatenación se produce igualmente y se rellenan con valores nulos los espacios donde no contamos con información:

In [4]:
df_norte["actualizado"] = True
df_sur["se_actualiza"] = False
df_ciudad_act_concat = pd.concat([df_norte, df_sur], ignore_index=True)
df_ciudad_act_concat

,ciudad,ventas,actualizado,se_actualiza
0,Oviedo,200,True,NaN
1,Avilés,150,True,NaN
2,A Coruña,180,True,NaN
3,Sevilla,120,NaN,False
4,Cádiz,140,NaN,False
5,Málaga,130,NaN,False
6,Córdoba,90,NaN,False


Análogamente podemos realizar concatenaciones horizontales con el mismo método agregando el argumento `axis=1`:

In [5]:
df_precio = pd.DataFrame({
    "precio": [10, 12, 11]
})
df_precio

,precio
0,10
1,12
2,11


In [6]:
df_ciudad_precio_concat = pd.concat([df_norte, df_precio], axis=1)
df_ciudad_precio_concat

,ciudad,ventas,actualizado,precio
0,Oviedo,200,True,10
1,Avilés,150,True,12
2,A Coruña,180,True,11


El procedimiento en caso de desigualdad entre DataFrames es análogo al anterior rellenando los huecos con valores perdidos:

In [7]:
df_ciudad_precio_sur_concat = pd.concat([df_sur, df_precio], axis=1)
df_ciudad_precio_sur_concat

,ciudad,ventas,se_actualiza,precio
0,Sevilla,120,False,10.0
1,Cádiz,140,False,12.0
2,Málaga,130,False,11.0
3,Córdoba,90,False,NaN


Además, es interesante observar que podemos concatenar tantos DataFrames como deseemos:

In [8]:
df_norte = pd.DataFrame({
    "ciudad": ["Oviedo", "Avilés", "A Coruña"],
    "ventas": [200, 150, 180]
})

df_sur = pd.DataFrame({
    "ciudad": ["Sevilla", "Cádiz", "Málaga", "Córdoba"],
    "ventas": [120, 140, 130, 90]
})

df_levante = pd.DataFrame({
    "ciudad": ["Barcelona", "Benidorm", "Girona"],
    "ventas": [110, 210, 160]
})

df_concat = pd.concat([df_norte, df_sur, df_levante], ignore_index=True)
df_concat

,ciudad,ventas
0,Oviedo,200
1,Avilés,150
2,A Coruña,180
3,Sevilla,120
4,Cádiz,140
5,Málaga,130
6,Córdoba,90
7,Barcelona,110
8,Benidorm,210
9,Girona,160


Cerraremos esta clase viendo cómo podemos gestionar los distintos tipos de unión mediante el argumento `join`.

Si indicamos `join="outer"` realizaremos una unión externa, es decir, conservaremos todas las columnas presentes en los DataFrames rellenando con nulos aquellos valores desconocidos. Esta es la opción programa por defecto

In [9]:
df_norte["actualizado"] = True
df_sur["se_actualiza"] = False
df_ciudad_act_concat_out = pd.concat([df_norte, df_sur], ignore_index=True, join="outer")
df_ciudad_act_concat_out

,ciudad,ventas,actualizado,se_actualiza
0,Oviedo,200,True,NaN
1,Avilés,150,True,NaN
2,A Coruña,180,True,NaN
3,Sevilla,120,NaN,False
4,Cádiz,140,NaN,False
5,Málaga,130,NaN,False
6,Córdoba,90,NaN,False


Sin embargo, si indicamos `join="inner"` la unión será interna y por tanto conservaremos únicamente las columnas comunes, aquellas que se encuentran presentes en ambos DataFrames:

In [10]:
df_ciudad_act_concat_in = pd.concat([df_norte, df_sur], ignore_index=True, join="inner")
df_ciudad_act_concat_in

,ciudad,ventas
0,Oviedo,200
1,Avilés,150
2,A Coruña,180
3,Sevilla,120
4,Cádiz,140
5,Málaga,130
6,Córdoba,90


### Ejercicios prácticos

* Dados los siguientes DataFrames:

In [11]:
import pandas as pd

data_madrid = {
    "edificio": [
        "Torre Picasso",
        "Torre de Cristal",
        "Torre Cepsa"
    ],
    "ciudad": [
        "Madrid",
        "Madrid",
        "Madrid"
    ],
    "altura_metros": [
        157,
        249,
        248
    ],
    "anio_construccion": [
        1988,
        2009,
        2009
    ]
}

df_madrid = pd.DataFrame(data_madrid)
display(df_madrid)


data_barcelona = {
    "edificio": [
        "Torre Glòries",
        "Hotel Arts Barcelona",
        "Torre Mapfre"
    ],
    "ciudad": [
        "Barcelona",
        "Barcelona",
        "Barcelona"
    ],
    "altura_metros": [
        144,
        154,
        154
    ],
    "anio_construccion": [
        2005,
        1994,
        1992
    ]
}

df_barcelona = pd.DataFrame(data_barcelona)
display(df_barcelona)

data_info = {
    "plantas": [45, 50, 49, 38, 44, 40,38],
    "uso": [
        "Oficinas",
        "Oficinas",
        "Oficinas",
        "Oficinas",
        "Hotel",
        "Oficinas",
        "Otros"
    ]
}

df_info = pd.DataFrame(data_info)
display(df_info)



,edificio,ciudad,altura_metros,anio_construccion
0,Torre Picasso,Madrid,157,1988
1,Torre de Cristal,Madrid,249,2009
2,Torre Cepsa,Madrid,248,2009


,edificio,ciudad,altura_metros,anio_construccion
0,Torre Glòries,Barcelona,144,2005
1,Hotel Arts Barcelona,Barcelona,154,1994
2,Torre Mapfre,Barcelona,154,1992


,plantas,uso
0,45,Oficinas
1,50,Oficinas
2,49,Oficinas
3,38,Oficinas
4,44,Hotel
5,40,Oficinas
6,38,Otros


- Combinar `df_madrid` y `df_barcelona` para crear un nuevo DataFrame llamado `df_edificios`:

In [12]:
df_edificios = pd.concat([df_madrid, df_barcelona])
df_edificios

,edificio,ciudad,altura_metros,anio_construccion
0,Torre Picasso,Madrid,157,1988
1,Torre de Cristal,Madrid,249,2009
2,Torre Cepsa,Madrid,248,2009
0,Torre Glòries,Barcelona,144,2005
1,Hotel Arts Barcelona,Barcelona,154,1994
2,Torre Mapfre,Barcelona,154,1992


- Realiza la misma concatenación reiniciando el índice:

In [13]:
df_edificios_reset_ind = pd.concat([df_madrid, df_barcelona], ignore_index=True)
df_edificios_reset_ind

,edificio,ciudad,altura_metros,anio_construccion
0,Torre Picasso,Madrid,157,1988
1,Torre de Cristal,Madrid,249,2009
2,Torre Cepsa,Madrid,248,2009
3,Torre Glòries,Barcelona,144,2005
4,Hotel Arts Barcelona,Barcelona,154,1994
5,Torre Mapfre,Barcelona,154,1992


- Añade las columnas de `df_info` al DataFrame anterior y comprueba las dimensiones finales:

In [14]:
df_completo = pd.concat([df_edificios_reset_ind, df_info], axis=1)
df_completo

,edificio,ciudad,altura_metros,anio_construccion,plantas,uso
0,Torre Picasso,Madrid,157.0,1988.0,45,Oficinas
1,Torre de Cristal,Madrid,249.0,2009.0,50,Oficinas
2,Torre Cepsa,Madrid,248.0,2009.0,49,Oficinas
3,Torre Glòries,Barcelona,144.0,2005.0,38,Oficinas
4,Hotel Arts Barcelona,Barcelona,154.0,1994.0,44,Hotel
5,Torre Mapfre,Barcelona,154.0,1992.0,40,Oficinas
6,NaN,NaN,NaN,NaN,38,Otros


En esta clase hemos estudiado la concatenación de DataFrames tanto en horizontal como en vertical. Es interesante ver que para la concatenación horizontal estamos asumiendo que la información se encuentra ordenada. En la siguiente clase estudiaremos el método `.merge()` que nos permite realizar esta operación si los datos no están ordenados o presentan duplicados.

## 8.2. Unión de DataFrames con `.merge()`

En esta clase vamos a explorar la unión de DataFrames mediante claves utilizando el método `.merge()` para combinar las tablas. Además exploraremos los distintos tipos de unión y cómo interpretar correctamente sus resultados.

### Introducción a `.merge()`

El método `.merge()` permite combinar DataFrames utilizando una o varias columnas comunes. Esto resulta especialmente útil cuando tenemos datos repartidos en distintas tablas y necesitamos combinarlos conservando la coherencia de la información.

Veamos un ejemplo de esta idea. Supongamos que tenemos de un lado la tabla con los datos de clientes:


In [15]:
import pandas as pd

clientes = pd.DataFrame({
    "cliente_id": [1, 2, 3, 4],
    "nombre": ["Ana", "Luis", "Carlos", "Marta"]
})

clientes

,cliente_id,nombre
0,1,Ana
1,2,Luis
2,3,Carlos
3,4,Marta


Y por otro lado tenemos las compras:

In [16]:
compras = pd.DataFrame({
    "cliente_id": [1, 2, 2, 3, 5],
    "producto": ["Portátil", "Tablet", "Monitor", "Teclado", "Auriculares"],
    "precio": [1200, 300, 200, 50, 120]
})

compras

,cliente_id,producto,precio
0,1,Portátil,1200
1,2,Tablet,300
2,2,Monitor,200
3,3,Teclado,50
4,5,Auriculares,120


Observando las tablas vemos que la columna `cliente_id` es la que nos permite saber que por ejemplo, Ana se ha comprado un portátil o Marta no se ha comprado nada.

Veamos cómo podemos usar `.merge()` para poder cruzar la información y obtener una tabla que la resuma:

In [17]:
data_cruzada = pd.merge(clientes, compras, on="cliente_id")
data_cruzada

,cliente_id,nombre,producto,precio
0,1,Ana,Portátil,1200
1,2,Luis,Tablet,300
2,2,Luis,Monitor,200
3,3,Carlos,Teclado,50


En este caso hemos obtenido una tabla que cruza la información de todos los clientes que compran generando por ejemplo un duplicado para Luis.

__Nota.__ Es importante observar que el cliente Marta no aparece en la segunda tabla porque se han conservado únicamente los registros que se encuentran presentes en ambas tablas. Esto se debe al tipo de join que hemos realizado.

### Tipos de join

El parámetros `how` nos permite definir el tipo de merge que deseamos realizar y viene por defecto a `inner`. Las posibles opciones son:

| Tipo    | Descripción                             |
| ------- | --------------------------------------- |
| `inner` | solo coincidencias                      |
| `left`  | todas las filas del DataFrame izquierdo |
| `right` | todas las filas del DataFrame derecho   |
| `outer` | todas las filas de ambos                |

__Nota.__ Esta operación y sus tipos es análoga al join presente en SQL.

Veamos la diferencia entre los tipos de merge en la práctica. Ya hemos visto cuál sería el resultado de un inner join en la ejecución anterior. Su opuesto sería un outer que conserva todas las filas de ambos DataFrames:

In [18]:
pd.merge(clientes, compras, how="outer")

,cliente_id,nombre,producto,precio
0,1,Ana,Portátil,1200.0
1,2,Luis,Tablet,300.0
2,2,Luis,Monitor,200.0
3,3,Carlos,Teclado,50.0
4,4,Marta,NaN,NaN
5,5,NaN,Auriculares,120.0


Observamos que los campos que no tienen correspondencia en la otra tabla se rellenan con valores nulos.

Si realizamos un merge left, conservaremos todos los campos de la primera tabla y solo los que tengan correspondencia de la segunda:

In [19]:
pd.merge(clientes, compras, how="left")

,cliente_id,nombre,producto,precio
0,1,Ana,Portátil,1200.0
1,2,Luis,Tablet,300.0
2,2,Luis,Monitor,200.0
3,3,Carlos,Teclado,50.0
4,4,Marta,NaN,NaN


Recíprocamente si realizamos un merge right conservaremos todos los campos de la segunda tabla y solo los de la primera con los que se establezcan correspondencias:

In [20]:
pd.merge(clientes, compras, how="right")

,cliente_id,nombre,producto,precio
0,1,Ana,Portátil,1200
1,2,Luis,Tablet,300
2,2,Luis,Monitor,200
3,3,Carlos,Teclado,50
4,5,NaN,Auriculares,120


### Otros parámetros relevantes

En ocasiones podemos encontrarnos que aunque la información se corresponda el campo por el que se cruza no tiene el mismo nombre, en ese caso podemos usar los argumentos `left_on`y `right_on` para que el método sea capaz de establecer la conexión:

In [21]:
antiguedad = pd.DataFrame({
    "num_cliente": [1, 2, 3, 4, 5],
    "meses": [36, 14, 100, 2, 60]
})

antiguedad

,num_cliente,meses
0,1,36
1,2,14
2,3,100
3,4,2
4,5,60


In [22]:
pd.merge(clientes, antiguedad, how="inner", left_on="cliente_id", right_on="num_cliente")

,cliente_id,nombre,num_cliente,meses
0,1,Ana,1,36
1,2,Luis,2,14
2,3,Carlos,3,100
3,4,Marta,4,2


Por último en ocasiones querremos realizar cruces por más de una columna. Esto es posible mediante el argumento `on` que puede tomar como valores un único elemento (una columna) o una lista de elementos (varias columnas):

In [23]:
df1 = pd.DataFrame({
    "ciudad": ["Madrid", "Madrid", "Barcelona", "Valencia"],
    "producto": ["Manzana", "Naranja", "Manzana", "Pera"],
    "ventas": [100, 150, 200, 120]
})

df1

,ciudad,producto,ventas
0,Madrid,Manzana,100
1,Madrid,Naranja,150
2,Barcelona,Manzana,200
3,Valencia,Pera,120


In [24]:
df2 = pd.DataFrame({
    "ciudad": ["Madrid", "Barcelona", "Barcelona", "Valencia"],
    "producto": ["Manzana", "Manzana", "Pera", "Pera"],
    "precio": [1.2, 1.1, 1.5, 1.3]
})
df2

,ciudad,producto,precio
0,Madrid,Manzana,1.2
1,Barcelona,Manzana,1.1
2,Barcelona,Pera,1.5
3,Valencia,Pera,1.3


In [25]:
pd.merge(df1, df2, on=["ciudad", "producto"], how="inner")

,ciudad,producto,ventas,precio
0,Madrid,Manzana,100,1.2
1,Barcelona,Manzana,200,1.1
2,Valencia,Pera,120,1.3


### Ejercicio práctico

* Dados los siguientes DataFrames:

In [26]:
df_temp = pd.DataFrame({
    "ciudad": ["Madrid", "Barcelona", "Sevilla", "Bilbao", "Valencia", "Málaga"],
    "pais": ["España", "España", "España", "España", "España", "España"],
    "fecha": ["2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10"],
    "temperatura_media": [8.5, 10.2, 12.8, 7.1, 11.0, 13.4]
})

df_temp

,ciudad,pais,fecha,temperatura_media
0,Madrid,España,2025-01-10,8.5
1,Barcelona,España,2025-01-10,10.2
2,Sevilla,España,2025-01-10,12.8
3,Bilbao,España,2025-01-10,7.1
4,Valencia,España,2025-01-10,11.0
5,Málaga,España,2025-01-10,13.4


In [27]:
df_lluvia = pd.DataFrame({
    "ciudad": ["Madrid", "Barcelona", "Sevilla", "Zaragoza", "Valencia", "Málaga"],
    "pais": ["España", "España", "España", "España", "España", "España"],
    "fecha": ["2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10"],
    "precipitacion_mm": [2.1, 0.0, 1.5, 0.3, 0.8, 0.0]
})

df_lluvia

,ciudad,pais,fecha,precipitacion_mm
0,Madrid,España,2025-01-10,2.1
1,Barcelona,España,2025-01-10,0.0
2,Sevilla,España,2025-01-10,1.5
3,Zaragoza,España,2025-01-10,0.3
4,Valencia,España,2025-01-10,0.8
5,Málaga,España,2025-01-10,0.0


In [28]:
df_humedad = pd.DataFrame({
    "localidad": ["Madrid", "Barcelona", "Bilbao", "Valencia", "Vigo", "Madrid", "Barcelona",],
    "fecha_medicion": ["2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-10", "2025-01-12", "2025-01-12"],
    "humedad_relativa": [71, 68, 82, 65, 79, 70, 67]
})

df_humedad

,localidad,fecha_medicion,humedad_relativa
0,Madrid,2025-01-10,71
1,Barcelona,2025-01-10,68
2,Bilbao,2025-01-10,82
3,Valencia,2025-01-10,65
4,Vigo,2025-01-10,79
5,Madrid,2025-01-12,70
6,Barcelona,2025-01-12,67


* Construye un DataFrame que contenga solo las ciudades para las que existen datos tanto de temperatura como de precipitación.

In [29]:
df_temp_lluv = pd.merge(df_temp, df_lluvia, how="inner")
df_temp_lluv

,ciudad,pais,fecha,temperatura_media,precipitacion_mm
0,Madrid,España,2025-01-10,8.5,2.1
1,Barcelona,España,2025-01-10,10.2,0.0
2,Sevilla,España,2025-01-10,12.8,1.5
3,Valencia,España,2025-01-10,11.0,0.8
4,Málaga,España,2025-01-10,13.4,0.0


* Crea un DataFrame que mantenga todas las ciudades del dataset de temperaturas, aunque no tengan datos de precipitación:

In [30]:
df_temp_lluv_l = pd.merge(df_temp, df_lluvia, how="left")
df_temp_lluv_l

,ciudad,pais,fecha,temperatura_media,precipitacion_mm
0,Madrid,España,2025-01-10,8.5,2.1
1,Barcelona,España,2025-01-10,10.2,0.0
2,Sevilla,España,2025-01-10,12.8,1.5
3,Bilbao,España,2025-01-10,7.1,NaN
4,Valencia,España,2025-01-10,11.0,0.8
5,Málaga,España,2025-01-10,13.4,0.0


* Construye un DataFrame que contenga todas las ciudades presentes en cualquiera de los dos datasets, aunque no haya datos en el otro.
Identifica qué ciudades tienen valores faltantes.

In [31]:
df_temp_lluv_o = pd.merge(df_temp, df_lluvia, how="outer")
df_temp_lluv_o

,ciudad,pais,fecha,temperatura_media,precipitacion_mm
0,Barcelona,España,2025-01-10,10.2,0.0
1,Bilbao,España,2025-01-10,7.1,NaN
2,Madrid,España,2025-01-10,8.5,2.1
3,Málaga,España,2025-01-10,13.4,0.0
4,Sevilla,España,2025-01-10,12.8,1.5
5,Valencia,España,2025-01-10,11.0,0.8
6,Zaragoza,España,2025-01-10,NaN,0.3


* Combina los datos de temperatura con los datos de humedad:

In [32]:
df_temp_hum = pd.merge(
    df_temp,
    df_humedad,
    left_on=["ciudad", "fecha"],
    right_on=["localidad", "fecha_medicion"]
)
df_temp_hum

,ciudad,pais,fecha,temperatura_media,localidad,fecha_medicion,humedad_relativa
0,Madrid,España,2025-01-10,8.5,Madrid,2025-01-10,71
1,Barcelona,España,2025-01-10,10.2,Barcelona,2025-01-10,68
2,Bilbao,España,2025-01-10,7.1,Bilbao,2025-01-10,82
3,Valencia,España,2025-01-10,11.0,Valencia,2025-01-10,65


En esta clase hemos explorado como `.merge()` nos permite unir DataFrames preservando la integridad de la información incluso si las columnas tienen duplicados o nombres distintos.

En la siguiente clase estudiaremos el método `.join()` para unir DataFrames a partir de la información almacenada en los índices.

## 8.3. Unión de dataframes con `.join()`

El método `.join()` permite combinar dos DataFrames utilizando los índices como clave de unión en lugar de una columna del DataFrame como en el caso del `.merge()`.

Este método es especialmente útil cuando los DataFrames ya están indexados y queremos añadir nuevas columnas.

Veamos a continuación un ejemplo de cómo podemos hacer esto en Pandas:


In [33]:
import pandas as pd

clientes = pd.DataFrame({
    "nombre": ["Ana", "Luis", "Carlos", "Marta"],
    "ciudad": ["Madrid", "Sevilla", "Valencia", "Madrid"]
}, index=["A189", "B145", "C643", "D759"])

clientes

,nombre,ciudad
A189,Ana,Madrid
B145,Luis,Sevilla
C643,Carlos,Valencia
D759,Marta,Madrid


In [34]:
compras = pd.DataFrame({
    "producto": ["Laptop", "Tablet", "Monitor"],
    "precio": [1200, 300, 200]
}, index=["A189", "B145", "C643"])

compras

,producto,precio
A189,Laptop,1200
B145,Tablet,300
C643,Monitor,200


Los índices en este caso representan códigos identificadores de clientes. Procedemos a unir los DataFrames con el índice como clave:

In [35]:
clientes.join(compras)

,nombre,ciudad,producto,precio
A189,Ana,Madrid,Laptop,1200.0
B145,Luis,Sevilla,Tablet,300.0
C643,Carlos,Valencia,Monitor,200.0
D759,Marta,Madrid,NaN,NaN


__Nota.__ Observamos que Marta no tiene ninguna compra registrada y aparece en la tabla resultado.

Esto se debe a que en el caso del método `.join()` disponemos de los mismos tipos de cruces que en `.merge()` pero en el caso de `.join()` el tipo elegido por defecto es `how="left"`.

Podemos explorar los distintos tipos de cruces igual que hicimos con `.merge()`:

In [36]:
clientes.join(compras, how="inner")

,nombre,ciudad,producto,precio
A189,Ana,Madrid,Laptop,1200
B145,Luis,Sevilla,Tablet,300
C643,Carlos,Valencia,Monitor,200


In [37]:
clientes.join(compras, how="right")

,nombre,ciudad,producto,precio
A189,Ana,Madrid,Laptop,1200
B145,Luis,Sevilla,Tablet,300
C643,Carlos,Valencia,Monitor,200


In [38]:
clientes.join(compras, how="outer")

,nombre,ciudad,producto,precio
A189,Ana,Madrid,Laptop,1200.0
B145,Luis,Sevilla,Tablet,300.0
C643,Carlos,Valencia,Monitor,200.0
D759,Marta,Madrid,NaN,NaN


La clave a la hora de elegir si debemos usar `.merge()` o `join()` consiste en la estructura de nuestros datos. `join()` requiere una determinada estructura y es menos flexible pero a cambio permite una sintaxis más limpia.

Además, resulta bastante habitual que el índice de un DataFrame sea una clave identificativa que permite cruzarlo con otros.

Es interesante ver que además podemos unir varios DataFrames en una misma sentencia:

In [39]:
antiguedad = pd.DataFrame({
    "meses": [60, 14, 26]
}, index=["A189", "B145", "C643"])

antiguedad

,meses
A189,60
B145,14
C643,26


In [40]:
df_join_completo = clientes.join([compras, antiguedad])
df_join_completo

,nombre,ciudad,producto,precio,meses
A189,Ana,Madrid,Laptop,1200.0,60.0
B145,Luis,Sevilla,Tablet,300.0,14.0
C643,Carlos,Valencia,Monitor,200.0,26.0
D759,Marta,Madrid,NaN,NaN,NaN


Por último, con `join()` también podemos cruzar utilizando una columna:

In [41]:
df1 = pd.DataFrame({
    "pedido_id": [101, 102, 103, 104],
    "cliente_id": [1, 2, 3, 2],
    "producto": ["Laptop", "Ratón", "Teclado", "Monitor"]
})
df1

,pedido_id,cliente_id,producto
0,101,1,Laptop
1,102,2,Ratón
2,103,3,Teclado
3,104,2,Monitor


In [42]:
df2 = pd.DataFrame({
    "cliente_id": [1, 2, 3],
    "nombre": ["Ana", "Luis", "Marta"],
    "ciudad": ["Madrid", "Barcelona", "Valencia"]
})
df2

,cliente_id,nombre,ciudad
0,1,Ana,Madrid
1,2,Luis,Barcelona
2,3,Marta,Valencia


In [43]:
df1.join(df2.set_index("cliente_id"), on="cliente_id")

,pedido_id,cliente_id,producto,nombre,ciudad
0,101,1,Laptop,Ana,Madrid
1,102,2,Ratón,Luis,Barcelona
2,103,3,Teclado,Marta,Valencia
3,104,2,Monitor,Luis,Barcelona


`df2.set_index("cliente_id")` convierte `cliente_id` en índice.

`join(..., on="cliente_id")` usa la columna `cliente_id` de df1 para buscar coincidencias en el índice de df2.

### Ejercicio práctico

* Dados los siguientes DataFrames:

In [44]:
df_peliculas = pd.DataFrame({
    "id_pelicula": [1, 2, 3, 4, 5],
    "titulo": [
        "Inception",
        "The Godfather",
        "Parasite",
        "The Dark Knight",
        "Pulp Fiction"
    ],
    "anio": [2010, 1972, 2019, 2008, 1994]
}).set_index("id_pelicula")

df_peliculas

,titulo,anio
id_pelicula,,
1,Inception,2010
2,The Godfather,1972
3,Parasite,2019
4,The Dark Knight,2008
5,Pulp Fiction,1994


In [45]:
df_directores = pd.DataFrame({
    "id_pelicula": [1, 2, 3, 4],
    "director": [
        "Christopher Nolan",
        "Francis Ford Coppola",
        "Bong Joon-ho",
        "Christopher Nolan"
    ]
}).set_index("id_pelicula")
df_directores

,director
id_pelicula,
1,Christopher Nolan
2,Francis Ford Coppola
3,Bong Joon-ho
4,Christopher Nolan


In [46]:
df_puntuacion = pd.DataFrame({
    "id_pelicula": [1, 2, 3, 4, 6],
    "puntuacion_imdb": [8.8, 9.2, 8.5, 9.0, 7.5]
}).set_index("id_pelicula")

df_puntuacion

,puntuacion_imdb
id_pelicula,
1,8.8
2,9.2
3,8.5
4,9.0
6,7.5


In [47]:
df_presupuesto = pd.DataFrame({
    "titulo": [
        "Inception",
        "The Godfather",
        "Parasite",
        "The Dark Knight",
        "Pulp Fiction"
    ],
    "presupuesto_millones": [160, 6, 11, 185, 8]
})
df_presupuesto

,titulo,presupuesto_millones
0,Inception,160
1,The Godfather,6
2,Parasite,11
3,The Dark Knight,185
4,Pulp Fiction,8


* Añade al DataFrame de películas la columna director:

In [48]:
df_join1 = df_peliculas.join(df_directores)
df_join1

,titulo,anio,director
id_pelicula,,,
1,Inception,2010,Christopher Nolan
2,The Godfather,1972,Francis Ford Coppola
3,Parasite,2019,Bong Joon-ho
4,The Dark Knight,2008,Christopher Nolan
5,Pulp Fiction,1994,NaN


* Añade ahora la columna puntuacion_imdb al DataFrame de películas:

In [49]:
df_join2 = df_peliculas.join(df_puntuacion)

df_join2

,titulo,anio,puntuacion_imdb
id_pelicula,,,
1,Inception,2010,8.8
2,The Godfather,1972,9.2
3,Parasite,2019,8.5
4,The Dark Knight,2008,9.0
5,Pulp Fiction,1994,NaN


* Crea un DataFrame que contenga películas, directores y puntuaciones combinando los tres datasets:

In [50]:
df_join3 = df_peliculas.join([df_directores, df_puntuacion])
df_join3

,titulo,anio,director,puntuacion_imdb
id_pelicula,,,,
1,Inception,2010.0,Christopher Nolan,8.8
2,The Godfather,1972.0,Francis Ford Coppola,9.2
3,Parasite,2019.0,Bong Joon-ho,8.5
4,The Dark Knight,2008.0,Christopher Nolan,9.0
5,Pulp Fiction,1994.0,NaN,NaN


* Crea un nuevo DataFrame que contenga solo las películas que tengan puntuación registrada:

In [51]:
df_join4 = df_peliculas.join(df_puntuacion, how="inner")
df_join4

,titulo,anio,puntuacion_imdb
id_pelicula,,,
1,Inception,2010,8.8
2,The Godfather,1972,9.2
3,Parasite,2019,8.5
4,The Dark Knight,2008,9.0


En esta clase hemos visto como `.join()` es el equivalente a `.merge()` cuando deseamos utilizar el índice como clave de unión. 

En la última clase de esta sección exploraremos distintos problemas que suelen surgir cuando se trabaja con estas uniones y posibles soluciones para ellos.

## 8.4. Resolución de conflictos en uniones 

Cuando trabajamos con métodos para la unión de distintos DataFrames afrontamos distintos problemas que pueden dificultar la interpretación del resultado final o incluso afectar a la coherencia de los datos resultantes. A continuación, recorreremos los principales problemas y algunas posibles soluciones para estas situaciones.

Partiremos de los siguientes DataFrames de ejemplo:

In [52]:
import pandas as pd

empleados = pd.DataFrame({
    "empleado_id": [1, 2, 3],
    "nombre": ["Ana", "Luis", "Carlos"],
    "ciudad": ["Madrid", "Sevilla", "Valencia"]
})

empleados

,empleado_id,nombre,ciudad
0,1,Ana,Madrid
1,2,Luis,Sevilla
2,3,Carlos,Valencia


In [53]:
salarios = pd.DataFrame({
    "empleado_id": [1, 2, 4],
    "ciudad": ["Madrid", "Sevilla", "Barcelona"],
    "salario": [2500, 3000, 2800]
})

salarios

,empleado_id,ciudad,salario
0,1,Madrid,2500
1,2,Sevilla,3000
2,4,Barcelona,2800


### Columnas duplicadas

Si realizamos un `.merge()` observamos que Pandas añade automáticamente el sufijo `_x` y `_y` a las columnas duplicadas para permitirnos distinguir el origen de cada una:

In [54]:
pd.merge(empleados, salarios, on="empleado_id")

,empleado_id,nombre,ciudad_x,ciudad_y,salario
0,1,Ana,Madrid,Madrid,2500
1,2,Luis,Sevilla,Sevilla,3000


La solución más evidente a este problema sería renombrar la columna en uno de los dos DataFrames:

In [55]:
salarios_renom = salarios.copy()
salarios_renom = salarios_renom.rename(columns={
    "ciudad" : "city"
})


In [56]:
salarios_renom

,empleado_id,city,salario
0,1,Madrid,2500
1,2,Sevilla,3000
2,4,Barcelona,2800


In [57]:
pd.merge(empleados, salarios_renom, on="empleado_id")

,empleado_id,nombre,ciudad,city,salario
0,1,Ana,Madrid,Madrid,2500
1,2,Luis,Sevilla,Sevilla,3000


Otra posibilidad es personalizar los sufijos para conservar los DataFrames iniciales pero tener trazabilidad en el DataFrame de unión:

In [58]:
pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    suffixes=("_empleado", "_salario")
)

,empleado_id,nombre,ciudad_empleado,ciudad_salario,salario
0,1,Ana,Madrid,Madrid,2500
1,2,Luis,Sevilla,Sevilla,3000


### Detección del origen de los valores

En ocasiones y sobre todo cuando no estamos tan acostumbrados a trabajar con la unión de DataFrames o no tenemos claro el tipo de unión que necesitaremos puede ser útil identificar el origen de cada registro del DataFrame resultante.

Para ello podemos usar el argumento `indicator=True` que crea una columna extra llamada `_merge` que nos indica si la fila resultante proviene de un DataFrame, del otro o de ambos:

In [59]:
pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    how="outer",
    indicator=True
)

,empleado_id,nombre,ciudad_x,ciudad_y,salario,_merge
0,1,Ana,Madrid,Madrid,2500.0,both
1,2,Luis,Sevilla,Sevilla,3000.0,both
2,3,Carlos,Valencia,NaN,NaN,left_only
3,4,NaN,NaN,Barcelona,2800.0,right_only


Esta función es además muy útil para llevar a cabo auditorías de datos, localizando por ejemplo, para qué registros no tenemos información en todas las tablas cuando sí deberíamos tenerlos.

### Claves duplicadas

La columna que utilicemos como clave de unión no tiene porque ser una columna de valores únicos. Es decir, podemos tener valores duplicados que conducirían a la generación de múltiples filas. Esto no es un problema per sé pero en ocasiones cuando trabajamos con datos de gran tamaño es importante tenerlo en cuenta para evitar que las tablas resultantes sean de un tamaño innecesariamente grande que puedan llegar a generar conflictos de recursos en su procesado.

Es una una buena práctica proceder al filtrado antes que al cruce para cruzar tablas lo más pequeñas posibles:

In [60]:
salarios = pd.DataFrame({
    "empleado_id": [1, 2, 2, 3, 3],
    "ciudad": ["Madrid", "Sevilla", "Sevilla", "Barcelona", "Barcelona"],
    "salario": [2500, 3000, 2800, 850, 900]
})

salarios

,empleado_id,ciudad,salario
0,1,Madrid,2500
1,2,Sevilla,3000
2,2,Sevilla,2800
3,3,Barcelona,850
4,3,Barcelona,900


In [61]:
cruce_empleado_salario = pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    how="left"
)
cruce_empleado_salario

,empleado_id,nombre,ciudad_x,ciudad_y,salario
0,1,Ana,Madrid,Madrid,2500
1,2,Luis,Sevilla,Sevilla,3000
2,2,Luis,Sevilla,Sevilla,2800
3,3,Carlos,Valencia,Barcelona,850
4,3,Carlos,Valencia,Barcelona,900


In [62]:
print(empleados.shape, cruce_empleado_salario.shape)

(3, 3) (5, 5)


Vemos que aún tratándose de un left join hemos obtenido un DataFrame con más registros que el DataFrame "izquierdo" original.

Para gestionar este tipo de situaciones contamos con el argumento `validate` que antes de realizar el merge comprueba si las claves son únicas o no.

Este argumento puede tomar los siguientes valores:

| opción         | significado                       |
| -------------- | --------------------------------- |
| `one_to_one`   | claves únicas en ambos DataFrames |
| `one_to_many`  | izquierda única                   |
| `many_to_one`  | derecha única                     |
| `many_to_many` | múltiples en ambos                |


In [63]:
pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    how="left", 
    validate="one_to_one"
)


MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  empleado_id
           2
           3 ...

In [64]:
pd.merge(
    empleados,
    salarios,
    on="empleado_id",
    how="left", 
    validate="one_to_many"
)


,empleado_id,nombre,ciudad_x,ciudad_y,salario
0,1,Ana,Madrid,Madrid,2500
1,2,Luis,Sevilla,Sevilla,3000
2,2,Luis,Sevilla,Sevilla,2800
3,3,Carlos,Valencia,Barcelona,850
4,3,Carlos,Valencia,Barcelona,900


Cuando vayamos a realizar cruces de DataFrames es recomendable:

* Revisar las claves antes de la unión para evitar duplicados innecesarios.
* Verificar claramente las relaciones entre tablas y añadir validadores si lo consideramos necesarios
* Usar `indicator=True` al principio cuando tengamos dudas con un merge determinado.
* Renombrar columnas conflictivas de forma directa o mediante el uso de sufijos.

### Ejercicio práctico

* Dados los siguientes DataFrames:

In [65]:
df_productos = pd.DataFrame({
    "id_producto": [101, 102, 103, 104, 105, 106],
    "producto": ["Manzana", "Leche", "Pan", "Arroz", "Yogur", "Tomate"],
    "categoria": ["Fruta", "Lácteo", "Panadería", "Cereal", "Lácteo", "Verdura"],
    "precio": [1.20, 0.95, 1.10, 2.30, 0.80, 1.50]
})
df_productos

,id_producto,producto,categoria,precio
0,101,Manzana,Fruta,1.20
1,102,Leche,Lácteo,0.95
2,103,Pan,Panadería,1.10
3,104,Arroz,Cereal,2.30
4,105,Yogur,Lácteo,0.80
5,106,Tomate,Verdura,1.50


In [66]:
df_ventas = pd.DataFrame({
    "id_producto": [101, 101, 102, 103, 103, 103, 107],
    "fecha": [
        "2025-01-10", "2025-01-11", "2025-01-10",
        "2025-01-10", "2025-01-11", "2025-01-12", "2025-01-10"
    ],
    "unidades_vendidas": [10, 8, 15, 20, 18, 25, 12],
    "precio": [1.20, 1.25, 0.95, 1.10, 1.10, 1.15, 2.00]
})
df_ventas

,id_producto,fecha,unidades_vendidas,precio
0,101,2025-01-10,10,1.20
1,101,2025-01-11,8,1.25
2,102,2025-01-10,15,0.95
3,103,2025-01-10,20,1.10
4,103,2025-01-11,18,1.10
5,103,2025-01-12,25,1.15
6,107,2025-01-10,12,2.00


In [67]:
df_proveedores = pd.DataFrame({
    "id_producto": [101, 101, 102, 104, 105, 108],
    "proveedor": ["Proveedor A", "Proveedor B", "Proveedor C", "Proveedor D", "Proveedor E", "Proveedor F"],
    "pais_origen": ["España", "Francia", "España", "Italia", "España", "Portugal"]
})

df_proveedores

,id_producto,proveedor,pais_origen
0,101,Proveedor A,España
1,101,Proveedor B,Francia
2,102,Proveedor C,España
3,104,Proveedor D,Italia
4,105,Proveedor E,España
5,108,Proveedor F,Portugal


In [68]:
df_stock = pd.DataFrame({
    "id_producto": [101, 102, 103, 104, 105],
    "stock": [50, 80, 40, 100, 60]
}).set_index("id_producto")
df_stock

,stock
id_producto,
101,50
102,80
103,40
104,100
105,60


* Combinar la información de ventas con el de productos y presta atención al tamaño del DataFrame resultante. Haz que el resultado permita distinguir el precio de cada tabla:

In [69]:
merge_1 = pd.merge(df_productos, df_ventas, on="id_producto", suffixes=("_catalogo", "_venta"))
merge_1

,id_producto,producto,categoria,precio_catalogo,fecha,unidades_vendidas,precio_venta
0,101,Manzana,Fruta,1.20,2025-01-10,10,1.20
1,101,Manzana,Fruta,1.20,2025-01-11,8,1.25
2,102,Leche,Lácteo,0.95,2025-01-10,15,0.95
3,103,Pan,Panadería,1.10,2025-01-10,20,1.10
4,103,Pan,Panadería,1.10,2025-01-11,18,1.10
5,103,Pan,Panadería,1.10,2025-01-12,25,1.15


* Combinar el dato de ventas con el de proveedores. Observa de nuevo el tamaño resultante:

In [70]:
merge_2 = pd.merge(df_ventas, df_proveedores, on="id_producto", how="inner")
print("Número de filas:", len(merge_2))
merge_2

Número de filas: 5


,id_producto,fecha,unidades_vendidas,precio,proveedor,pais_origen
0,101,2025-01-10,10,1.20,Proveedor A,España
1,101,2025-01-10,10,1.20,Proveedor B,Francia
2,101,2025-01-11,8,1.25,Proveedor A,España
3,101,2025-01-11,8,1.25,Proveedor B,Francia
4,102,2025-01-10,15,0.95,Proveedor C,España


* Une los datos de producto y stock usando `.join()`:

In [71]:
df_productos_idx = df_productos.set_index("id_producto")

join_3 = df_productos_idx.join(df_stock)
join_3

,producto,categoria,precio,stock
id_producto,,,,
101,Manzana,Fruta,1.20,50.0
102,Leche,Lácteo,0.95,80.0
103,Pan,Panadería,1.10,40.0
104,Arroz,Cereal,2.30,100.0
105,Yogur,Lácteo,0.80,60.0
106,Tomate,Verdura,1.50,NaN


La unión de DataFrames mediante distintas técnicas es un problemas bastante habitual al trabajar con datos y es importante comprender las implicaciones que puede tener elegir un método u otro así como las posibles alteraciones en los datos que pueden generar estos cruces.

En las última sección haremos un repaso general de las principales diferencias entre NumPy y Pandas para cerrar este curso.